In [ ]:
# ---------- Imports ----------
import os, json, random, numpy as np
import pandas as pd, matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score, confusion_matrix

In [ ]:
# ---------- CONFIG ----------
MODEL_ID = "distilgpt2"        # small default for dry-run. Replace with 'mistral-small-24b' or 'mosaicml/gemma-3-27b' if you have resources.
DRY_RUN = True                 # True -> quick synthetic features; False -> attempt to load MODEL_ID
SELECT_DATASETS = ["ID", "ST"] # keys (or local jsonl paths) - synthetic data created if file not found
MAX_EXAMPLES_PER_DATASET = 300
LAYER_TO_EXTRACT = -1
PROBE_C = 1.0
TEST_SIZE = 0.2
RANDOM_SEED = 42
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"

# Upper-bound options:
# 'merged_inclusive': train probe on union of all datasets' training splits (privileged labels available)
# 'leave_one_out'    : for each dataset, train on union of *other* datasets' training splits
UPPER_BOUND_MODE = "merged_inclusive"  # or "leave_one_out"
TRAIN_UPPER_BOUND = True               # whether to run the upper-bound experiments

# set seed
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# ---------- dataset loader ----------
def load_datasets(keys, max_per_dataset=200):
    datasets = {}
    for k in keys:
        if os.path.exists(k):
            # load jsonl
            arr = []
            with open(k, "r", encoding="utf-8") as fh:
                for i, line in enumerate(fh):
                    if i >= max_per_dataset:
                        break
                    try:
                        obj = json.loads(line)
                        text = obj.get("prompt") or obj.get("text") or obj.get("context") or obj.get("input") or str(obj)
                        label = obj.get("label") if "label" in obj else obj.get("is_lie") if "is_lie" in obj else None
                        if label is None:
                            label = obj.get("y") or obj.get("lie") or obj.get("honest") or 0
                        if isinstance(label, bool):
                            label = int(label)
                        arr.append({"text": text, "label": int(label)})
                    except Exception:
                        continue
            print(f"Loaded {len(arr)} examples from file {k}")
            datasets[k] = arr
        else:
            # synthetic alternating examples
            arr = []
            for i in range(max_per_dataset):
                lbl = i % 2
                txt = f"SYN-{k}-{i}: This is {'a lie' if lbl==1 else 'honest'} response."
                arr.append({"text": txt, "label": lbl})
            print(f"Created synthetic dataset {k} with {len(arr)} examples")
            datasets[k] = arr
    return datasets

In [ ]:
# ---------- Model load & pad_token fix ----------
def safe_load_model(model_id, device="cpu", try_fp16=True):
    """
    Attempts to load a causal LM. Fixes tokenizer.pad_token if missing.
    Returns tokenizer, model (or (None, None) on failure).
    """
    try:
        # load tokenizer first
        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)
        # If tokenizer has no pad_token, try to set it to eos_token; else add '[PAD]'
        added_special = False
        if tokenizer.pad_token is None:
            if getattr(tokenizer, "eos_token", None) is not None:
                tokenizer.pad_token = tokenizer.eos_token
                print("Tokenizer had no pad_token → set pad_token = eos_token")
            else:
                tokenizer.add_special_tokens({"pad_token": "[PAD]"})
                added_special = True
                print("Tokenizer had no pad_token or eos_token → added [PAD] as pad_token")

        # to load the model
        kwargs = {"output_hidden_states": True, "trust_remote_code": True}
        if device == "cuda" and try_fp16:
            model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto", **kwargs)
        else:
            model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
            if device == "cuda":
                model.to("cuda")
        # incase we added a special token to tokenizer, resize embeddings
        if added_special:
            model.resize_token_embeddings(len(tokenizer))
        # also ensure tokenizer.padding_side exists
        if not hasattr(tokenizer, "padding_side"):
            tokenizer.padding_side = "left"
        print(f"Loaded model/tokenizer: {model_id}")
        return tokenizer, model
    except Exception as e:
        print("Model load failed:", e)
        return None, None

In [ ]:
# ---------- Activation extraction ----------
def get_mean_activations(tokenizer, model, texts, layer_index=-1, device="cpu", max_length=512, batch_size=8):
    """
    Returns shape (n_texts, hidden_size) of mean-pooled hidden states at layer_index.
    If model/tokenizer None -> raises.
    """
    model.eval()
    dev = torch.device(device if (torch.cuda.is_available() and device=="cuda") else "cpu")
    all_vecs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(dev)
            # forward
            out = model(**enc, output_hidden_states=True)
            # prefer (hidden_states); fallback to last_hidden_state if needed
            if hasattr(out, "hidden_states") and out.hidden_states is not None:
                hs = out.hidden_states[layer_index]  # (bs, seq_len, hidden)
            elif hasattr(out, "last_hidden_state"):
                hs = out.last_hidden_state
            else:
                raise RuntimeError("Model output has no hidden states or last_hidden_state.")
            mask = enc.attention_mask.unsqueeze(-1)
            masked = hs * mask
            summed = masked.sum(dim=1)
            lens = mask.sum(dim=1).clamp(min=1)
            meanpooled = summed / lens
            all_vecs.append(meanpooled.cpu().numpy())
    return np.vstack(all_vecs)

In [ ]:
# ---------- Probe training & metrics ----------
def train_probe(X_train, y_train, C=1.0, random_state=42):
    clf = LogisticRegression(C=C, solver="liblinear", max_iter=2000, random_state=random_state)
    clf.fit(X_train, y_train)
    return clf

def eval_probe(clf, X_test, y_test, thresh=0.5):
    probs = clf.predict_proba(X_test)[:,1]
    preds = (probs >= thresh).astype(int)
    auroc = roc_auc_score(y_test, probs) if len(set(y_test))>1 else float("nan")
    bacc = balanced_accuracy_score(y_test, preds)
    recall_pos = recall_score(y_test, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return {"auroc": float(auroc), "balanced_acc": float(bacc), "recall_pos": float(recall_pos), "fpr": float(fpr)}

In [ ]:
# ---------- Experiment runner with upper-bound variants ----------
def run_experiments(datasets, model_id=None, dry_run=True, device="cpu", layer_index=-1, probe_C=1.0, test_size=0.2, random_seed=42,
                    train_upper_bound=True, upper_bound_mode="merged_inclusive"):
    random.seed(random_seed)
    np.random.seed(random_seed)
    tokenizer = model = None
    # attempt to load model if not dry-run
    if not dry_run and model_id:
        print("Attempting to load model:", model_id)
        tokenizer, model = safe_load_model(model_id, device=device)
        if tokenizer is None:
            print("Model unavailable; will fallback to synthetic numeric features.")
    results = {}
    # Per-dataset: split into train/test, then train probe on dataset train and test on dataset test
    splits = {}
    for key, examples in datasets.items():
        texts = [ex["text"] for ex in examples]
        labels = np.array([int(ex["label"]) for ex in examples])
        if len(set(labels)) < 2:
            print(f"Skipping {key}: only one label present.")
            continue
        # split
        X_indices = np.arange(len(texts))
        # maintain stratify if possible
        strat = labels if len(set(labels))>1 else None
        train_idx, test_idx = train_test_split(X_indices, test_size=test_size, random_state=random_seed, stratify=strat)
        splits[key] = {"train_idx": train_idx, "test_idx": test_idx, "texts": texts, "labels": labels}

    # Pre-extract activations per dataset (or fallback synthetic)
    activations = {}
    for key, info in splits.items():
        try:
            if tokenizer is None or model is None:
                raise RuntimeError("Model/tokenizer unavailable (dry-run).")
            # extract all activations for this dataset
            A = get_mean_activations(tokenizer, model, info["texts"], layer_index=layer_index, device=device)
            print(f"Extracted activations for {key}: {A.shape}")
        except Exception as e:
            print(f"Activation extraction failed for {key}:", e)
            print(" -> Falling back to synthetic random features for this dataset.")
            A = np.random.normal(size=(len(info["texts"]), 768))
        activations[key] = A

    # Train per-dataset probes (realistic mean-probe)
    for key, info in splits.items():
        A = activations[key]
        y = info["labels"]
        train_i = info["train_idx"]; test_i = info["test_idx"]
        X_tr = A[train_i]; X_te = A[test_i]; y_tr = y[train_i]; y_te = y[test_i]
        clf = train_probe(X_tr, y_tr, C=probe_C, random_state=random_seed)
        metrics = eval_probe(clf, X_te, y_te)
        results[key] = metrics
        print(f"[MEAN PROBE] {key}: {metrics}")

    # UPPER-BOUND: privileged-data probe
    if train_upper_bound and len(splits) > 0:
        if upper_bound_mode == "merged_inclusive":
            # Build merged training set by concatenating all datasets' TRAIN splits (privileged data)
            X_tr_list = []; y_tr_list = []
            for key, info in splits.items():
                A = activations[key]; y = info["labels"]
                tr = info["train_idx"]
                X_tr_list.append(A[tr]); y_tr_list.append(y[tr])
            X_train_merged = np.vstack(X_tr_list); y_train_merged = np.concatenate(y_tr_list)
            print("Upper-bound (merged_inclusive): merged training size:", X_train_merged.shape[0])
            # Train one probe on merged training, evaluate on each dataset's TEST split
            clf_ub = train_probe(X_train_merged, y_train_merged, C=probe_C, random_state=random_seed)
            for key, info in splits.items():
                A = activations[key]; y = info["labels"]; test_i = info["test_idx"]
                metrics = eval_probe(clf_ub, A[test_i], y[test_i])
                results[f"UPPER_MERGED->{key}"] = metrics
                print(f"[UPPER_MERGED] test on {key}: {metrics}")

        elif upper_bound_mode == "leave_one_out":
            # for each dataset: train on union of other datasets' TRAIN splits and test on that dataset's TEST split
            for target_key in splits.keys():
                X_tr_list=[]; y_tr_list=[]
                for key, info in splits.items():
                    if key == target_key: continue
                    A = activations[key]; y = info["labels"]; tr = info["train_idx"]
                    X_tr_list.append(A[tr]); y_tr_list.append(y[tr])
                if len(X_tr_list)==0:
                    print("No other datasets to train on for leave_one_out; skipping", target_key)
                    continue
                X_train_loo = np.vstack(X_tr_list); y_train_loo = np.concatenate(y_tr_list)
                clf_ub = train_probe(X_train_loo, y_train_loo, C=probe_C, random_state=random_seed)
                # test on target's test split
                A_target = activations[target_key]; y_target = splits[target_key]["labels"]; test_i = splits[target_key]["test_idx"]
                metrics = eval_probe(clf_ub, A_target[test_i], y_target[test_i])
                results[f"UPPER_LOO->{target_key}"] = metrics
                print(f"[UPPER_LOO] test on {target_key}: {metrics}")
        else:
            print("Unknown upper_bound_mode:", upper_bound_mode)
    return results

In [ ]:
# ---------- Reporting ----------
def report_results(results):
    if len(results)==0:
        print("No results.")
        return
    df = pd.DataFrame.from_dict(results, orient="index")
    df.loc["MEAN"] = df.mean(axis=0)
    print("\nResults table (with MEAN row):")
    display(df)
    metrics = ["auroc", "balanced_acc", "recall_pos", "fpr"]
    keys = list(results.keys())
    arr = np.array([[results[k][m] for m in metrics] for k in keys])
    x = np.arange(len(keys))
    width = 0.18
    plt.figure(figsize=(12,5))
    for i, m in enumerate(metrics):
        plt.bar(x + (i-1.5)*width, arr[:,i], width, label=m)
    plt.xticks(x, keys, rotation=45, ha="right")
    plt.ylim(0,1)
    plt.title("Probe metrics by dataset (FPR lower better; others higher better)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------- Run ----------
if __name__ == "__main__":
    ds = load_datasets(SELECT_DATASETS, max_per_dataset=MAX_EXAMPLES_PER_DATASET)
    res = run_experiments(ds, model_id=MODEL_ID, dry_run=DRY_RUN, device=DEVICE,
                          layer_index=LAYER_TO_EXTRACT, probe_C=PROBE_C, test_size=TEST_SIZE,
                          random_seed=RANDOM_SEED, train_upper_bound=TRAIN_UPPER_BOUND,
                          upper_bound_mode=UPPER_BOUND_MODE)
    report_results(res)